# 9. Opening the pickle

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

The official right hand is `mano_v1_2/models/MANO_RIGHT.pkl`. It is a dictionary saved with chumpy. `official_mano.py` turns the arrays into NumPy. chumpy itself does not have to be installed.

The licence does not allow those pickles to be committed to a public repository. The `.gitignore` in this folder ignores `mano_v1_2/models/*.pkl`. The files stay on this machine. The code only reads them.

## The keys the forward formula uses

| Key | Shape in this file | Role in the formula |
|---|---|---|
| `v_template` | (778, 3) | Mean hand, metres |
| `f` | (1538, 3) | Triangle vertex indices. They do not change with pose |
| `shapedirs` | 20 components are stored; the first 10 are read, giving (778, 3, 10) | Shape displacements |
| `posedirs` | (778, 3, 135) | Pose blend. 15 joints, nine numbers of `R - I` each |
| `J_regressor` | (16, 778) | A sparse matrix. In this right-hand file the weights are nonnegative and each row sums to 1 |
| `weights` | (778, 16) | Skinning weights. Each row sums to 1 |
| `kintree_table` | (2, 16) | Row 0 is the parent. The root parent is often stored as the unsigned integer 4294967295, which is -1 |
| `hands_components` | (45, 45) | PCA basis of the finger axis-angles |
| `hands_mean` | (45,) | Mean finger pose of the training registrations |

There is also `hands_coeffs`, the PCA variance spectrum. The forward pass does not multiply by it.

The next cell prints these shapes and checks the parents, the skinning weights, and the joint regressor.


In [1]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

import mano_skeleton as sk
from official_mano import require_model

model = require_model()
rows = [
    ("v_template", model.v_template.shape),
    ("shapedirs", model.shapedirs.shape),
    ("posedirs", model.posedirs.shape),
    ("j_regressor", model.j_regressor.shape),
    ("weights", model.weights.shape),
    ("faces", model.faces.shape),
    ("hands_components", model.hands_components.shape),
    ("hands_mean", model.hands_mean.shape),
]
for name, shape in rows:
    print(f"{name:<18} {shape}")

print("parents", model.parents.tolist())
print("skinning row sums  min/max", float(model.weights.sum(1).min()), float(model.weights.sum(1).max()))
print("J row sums", np.round(model.j_regressor.sum(1), 4))
print("J minimum", float(model.j_regressor.min()))

assert model.v_template.shape == (778, 3)
assert model.shapedirs.shape == (778, 3, 10)
assert model.posedirs.shape == (778, 3, 135)
assert model.j_regressor.shape == (16, 778)
assert model.weights.shape == (778, 16)
assert model.faces.shape == (1538, 3)
assert np.array_equal(model.parents, sk.PARENTS)
assert np.allclose(model.weights.sum(1), 1.0)
assert np.allclose(model.j_regressor.sum(1), 1.0)
assert model.j_regressor.min() >= 0.0


v_template         (778, 3)
shapedirs          (778, 3, 10)
posedirs           (778, 3, 135)
j_regressor        (16, 778)
weights            (778, 16)
faces              (1538, 3)
hands_components   (45, 45)
hands_mean         (45,)
parents [-1, 0, 1, 2, 0, 4, 5, 0, 7, 8, 0, 10, 11, 0, 13, 14]
skinning row sums  min/max 0.9999999999999998 1.0000000000000002
J row sums [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
J minimum 0.0


## Beside the toy

| Toy | Official |
|---|---|
| 5 vertices, 4 joints | 778 vertices, 16 joints |
| 1 shape component | 10 usable shape components |
| 27-D pose feature | 135-D pose feature |
| joint i = vertex i | a joint is a weighted average of 778 vertices |
| one skin vertex with weights (0.25, 0.75) | one row of 16 weights per vertex |

The formula is unchanged. What changes is the size of the arrays, and the axes of the official template. The next chapter measures which way this right hand points.
